# M0 — independent Gaussian copula

This is a self-contained scientific monograph for M0. It assumes no prior notebook. See [the research problem](../../docs/scientific/01_research_problem.md), [finite PIT construction](../../docs/scientific/03_chronos_and_pit.md), and [dependence models](../../docs/scientific/04_dependence_models.md).

## Complete notation and validity

$i$ indexes forecast instance with origin $t^{(i)}$; $g$ indexes a static group $\mathcal E_g=\{k_1,\ldots,k_{K_g}\}$; and $\tau$ is one-based lead. $Y_{k,\tau}^{(i)}$ is random, $y_{k,\tau}^{(i)}$ is observed, and $F_{k,\tau}^{(i)}$ is its fixed marginal forecast. $A_{g,\tau}^{(i)}=\sum_kY_{k,\tau}^{(i)}$ is the random aggregate.

$V_{k,\tau}^{(i)}=1$ means entity $k$ has a finite observation and valid marginal grid. The complete-group indicator is $V_{g,\tau}^{(i)}=\prod_{k\in\mathcal E_g}V_{k,\tau}^{(i)}$. M0 is evaluated only if $V_g=1$; it never removes an invalid entity.

## Statistical hypothesis

M0 asserts conditional rank independence, not independence of physical loads:
$$C_{g,\tau}^{(i)}(u_1,\ldots,u_{K_g})=\prod_{k=1}^{K_g}u_k,\qquad R_{g,\tau}^{(i)}=I_{K_g}.$$
Draw $\eta^{(m)}\sim\mathcal N(0,I_{K_g})$ and set $U_k^{(m)}=\Phi(\eta_k^{(m)})$. The components are independent uniforms. Projecting them to each fixed marginal and summing yields $\widetilde A^{(m)}=\sum_k\widetilde Y_k^{(m)}$. M0 therefore asks what aggregate uncertainty would remain if residual ranks had no spatial dependence.

In [ ]:
from pathlib import Path
import sys, numpy as np, matplotlib.pyplot as plt
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(PROJECT_ROOT / 'src') not in sys.path: sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from simcast.config import load_base_config, load_method_config, resolve_run_config
from simcast.experiments import locate_compatible_cache
from simcast.fm.cache import load_pit_library
from simcast.cli.train_dependence import train_from_config
from simcast.cli.evaluate import evaluate_from_config
BASE_FILE='configs/bases/liander2024/transformer.yaml'
METHOD_FILE='configs/methods/m0_independent.yaml'
BASE_OVERRIDES=(); METHOD_OVERRIDES=()
TRAIN_IF_MISSING=False; RUN_EVALUATION=False
RUN_DIR=PROJECT_ROOT/'runs'/'notebook_walkthrough'/'m0_independent'
base=load_base_config(PROJECT_ROOT/BASE_FILE,overrides=BASE_OVERRIDES)
method=load_method_config(PROJECT_ROOT/METHOD_FILE,overrides=METHOD_OVERRIDES)
config=resolve_run_config(base,method)
cache_dir=locate_compatible_cache(base)
library=load_pit_library(cache_dir,access='training'); ds=library.dataset
entity_ids=[str(x) for x in ds.entity_id.values]; K_g=len(entity_ids)
print({'g':base.data.entity_type,'K_g':K_g,'entities':entity_ids,'method':method.id})


## Correlation and aggregate consequence

For two components with marginal variances $\sigma_1^2,\sigma_2^2$, $\mathrm{Var}(Y_1+Y_2)=\sigma_1^2+\sigma_2^2+2\mathrm{Cov}(Y_1,Y_2)$. M0 removes the residual-rank covariance term. If residual ranks are positively associated, M0 commonly understates aggregate dispersion even though every entity marginal is correct.

In [ ]:
R=np.eye(K_g)
fig,ax=plt.subplots(figsize=(5,4)); im=ax.imshow(R,vmin=-1,vmax=1,cmap='coolwarm')
ax.set(title=fr'M0: $R=I_{{{K_g}}}$',xlabel='entity index',ylabel='entity index'); fig.colorbar(im,ax=ax,label='correlation'); plt.show()


## Optional authoritative computation

The following cell is inert by default. Enabling training calls the same `train_from_config` used by the CLI; for M0, “training” only records the identity model. Evaluation opens sealed test labels and writes scores, so enable it only for a new, deliberately named analysis directory.

In [ ]:
if TRAIN_IF_MISSING and not (RUN_DIR/'model.npz').is_file():
    train_from_config(config,cache_dir=cache_dir,output_dir=RUN_DIR)
if RUN_EVALUATION:
    evaluate_from_config(config,methods=('independent',),method_runs={'independent':RUN_DIR},cache_dir=cache_dir,output_dir=RUN_DIR/'evaluation')


## Interpretation and controls

M0 has no fitted hyperparameters: its complete method file contains only `kind`, `id`, and `family: independent`. In the base file, `sampling.num_samples` determines Monte Carlo resolution, `sampling.common_random_numbers` makes comparisons paired, and `evaluation.*` defines scores. `IndependentCopula` returns $I_{K_g}$ and the Gaussian-copula sampler generates joint ranks. Poor M0 aggregate calibration is evidence of residual spatial dependence, not a causal claim.